# 02. Protocol-fixed baseline pipeline

**???? ???????.** ??????? ?????????? baseline-?????? ?? ??? ?? ?????????, ??? ? ????????? Efficient HADES: `stratified_by_workload`, horizon targets, validation-selected thresholds ? ?????????? ??????? F1/Precision/Recall/AP/ROC-AUC.

Baseline ????? ?? ???????? ?????? ?????? ?????????. ??? ??????? ??????????? ?????, ??????? ??????????, ??? ???? preprocessing ? data protocol ?? ??????? ??????????????? ???????????.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "dashboard" / "artifacts"
CONTRACT_PATH = ARTIFACT_DIR / "contract" / "strict_intensity_contract.csv.gz"
REPORT_DIR = ARTIFACT_DIR / "reports"
BASELINE_DIR = REPORT_DIR / "protocol_fixed_baselines"
MODEL_OUT_DIR = ARTIFACT_DIR / "model" / "protocol_fixed_efficient_hades"
RUN_ROOT = ARTIFACT_DIR / "model_runs" / "protocol_fixed_hades"
RUN_ID = "final_protocol_fixed_hades"
BUNDLE_DIR = ARTIFACT_DIR / "bundle"

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def display_existing(path: Path, reader=pd.read_csv, **kwargs):
    if not path.exists():
        print(f"???? ???? ?? ??????: {path.relative_to(PROJECT_ROOT)}")
        return pd.DataFrame()
    frame = reader(path, **kwargs)
    display(frame)
    return frame


## 1. ???? baseline

**???? ???????.** ?????????, ??? baseline ????? ?????? ?????? ?? strict contract. ???? contract ???????????, notebook ?????? ??? ??? ?? ???????, ??? ? ????????? ??????.


In [ ]:
if not CONTRACT_PATH.exists():
    run_command([sys.executable, "-m", "dashboard.pipeline.build_contract", "--raw-data", "raw_data"])
contract = pd.read_csv(CONTRACT_PATH, low_memory=False)
print(f"Contract rows: {len(contract):,}")
display(contract.groupby("split").agg(rows=("run_id", "size"), runs=("run_id", "nunique"), workloads=("workload", "nunique")).reset_index())


**????????? ?????.** Baseline ? ????????? ?????? ???????? ?? ????? ???????. ??? ??????? ?????? ????????, ????? ?????? ???????????? ?????????? ?????? ?????? ??????.


## 2. ???????? baseline

**???? ???????.** ????????? protocol-fixed baseline. ?? ?????????? log-only, metric-only, metric raw+deviation ? fused log+metric ???????? ?? ?????????? targets.


In [ ]:
TRAIN_BASELINES = True
baseline_command = [
    sys.executable, "-m", "dashboard.training.baselines",
    "--contract-path", str(CONTRACT_PATH),
    "--output-dir", str(BASELINE_DIR),
    "--split-protocol", "stratified_by_workload",
    "--targets", "default",
    "--seed", "43",
    "--validation-seed", "43",
    "--fp-budget", "0.10",
]
if TRAIN_BASELINES:
    run_command(baseline_command)
else:
    print("Baseline training skipped; existing artifacts will be read.")


**????????? ?????.** `Run All` ??????????? baseline. ??? ??????? ????????? ???????? ????? ???????? ????????? `TRAIN_BASELINES = False`, ?? ????????? ????? ?????? ????????? ????? ??????? ???????.


## 3. Window-level ???????

**???? ???????.** ?????????? ???????? ??????? ?? test: F1, Precision, Recall, AP ? ROC-AUC. ???????? ????? ?????????? ?? validation, ? test ?? ???????????? ??? ??????? threshold.


In [ ]:
metrics_path = BASELINE_DIR / "baseline_protocol_metrics.csv"
metrics = pd.read_csv(metrics_path)
test_best = metrics.query("split == 'test' and threshold_policy == 'validation_best_f1'").copy()
score_cols = ["average_precision", "roc_auc", "precision", "recall", "f1", "false_positives", "missed_anomalies"]
for col in score_cols:
    if col in test_best.columns:
        test_best[col] = pd.to_numeric(test_best[col], errors="coerce")

best_by_target = test_best.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)
display(best_by_target[["target", "baseline", "precision", "recall", "f1", "average_precision", "roc_auc", "false_positives", "missed_anomalies"]].round(4))


In [ ]:
plot_targets = ["horizon_60s", "horizon_120s", "horizon_300s", "y_warning", "y_risk"]
plot_df = best_by_target[best_by_target["target"].isin(plot_targets)].set_index("target")[["precision", "recall", "f1"]]
ax = plot_df.plot(kind="bar", figsize=(10, 4), title="?????? baseline ?? target: Precision / Recall / F1")
ax.set_xlabel("Target")
ax.set_ylabel("???????? ???????")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()


**????????? ?????.** Baseline ??????????, ????? ????????? ??????? ???????? ?? ???????????? fusion-??????. ???? baseline ????? ?? `horizon_300s`, ????????? ?????? ?????? ?????????? ?? ?????? window-F1, ?? ? ?????????????, alert-layer ? ??????????????? ??????????.


## 4. FP-budget ?????

**???? ???????.** ????????? ????? ????? ????? ??????. ??????? recall ??? ?? ???? ?? ?????, ???? ?????? ??????? ????? ?????? ?? ??????; ??????? ????????????? ??????? precision/recall ??? ??????????? false positives.


In [ ]:
fp_view = metrics.query("split == 'test' and threshold_policy.str.contains('fp_budget', regex=False)", engine="python").copy()
fp_best = fp_view.sort_values(["target", "f1"], ascending=[True, False]).groupby("target", as_index=False).head(1)
display(fp_best[["target", "baseline", "threshold_policy", "precision", "recall", "f1", "false_positives", "missed_anomalies"]].round(4))


**????????? ?????.** FP-budget ?? ?????? ??????????????? F1. ?? ????? ??? ??????????? ?????: ?????? ?????? ??????, ?? ???????? ????????. ??? ????? ???????? ???????????? ? alert-layer ? `03`.


## 5. Breakdown ?? workload

**???? ???????.** ?????????, ?? ???????? ?? ???????? ? ??????? ????????. ??? ??? ????? ????????, ??? baseline ???????? ?????????, ? ??? workload ???????? ?????? ??????.


In [ ]:
workload_path = BASELINE_DIR / "baseline_protocol_workload_metrics.csv"
workload_metrics = pd.read_csv(workload_path) if workload_path.exists() else pd.DataFrame()
if not workload_metrics.empty:
    best_keys = best_by_target[["target", "baseline"]].drop_duplicates()
    workload_best = workload_metrics.merge(best_keys, on=["target", "baseline"], how="inner")
    display(workload_best[["target", "baseline", "workload", "precision", "recall", "f1", "false_positives", "missed_anomalies"]].round(4))
else:
    print("Workload metrics ?? ???????.")


**????????? ?????.** Workload breakdown ???????? ?? ????????? ?????? ?? ????? ??????? F1. ???? ?????????? workload ?????, ??? ???????? protocol/preprocessing/??????, ? ?? ??????????? ???????????.


## ???? 02

Baseline stage ????????? ??????????? ????? ??? ????????? ??????: ?????????? split, targets, threshold policy ? ???????. ??? artifacts ???????? ? `03_fused_model.ipynb` ??? ??????? ????????? ? Efficient HADES.
